# Ethiopia Climate Data Analysis
Exploratory data analysis for Ethiopia climate dataset.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load the Ethiopia climate data
df = pd.read_csv('ethiopia.csv')

# Display basic info
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Check data types and missing values
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
df.describe()

In [ ]:
# Parse dates - YEAR and DOY (Day of Year) to datetime
df['DATE'] = pd.to_datetime(df['YEAR'].astype(str) + df['DOY'].astype(str).str.zfill(3), format='%Y%j')

print("Date Range:")
print(f"Start: {df['DATE'].min()}")
print(f"End: {df['DATE'].max()}")
print(f"\nTotal days: {len(df)}")

# Display with new date column
df[['YEAR', 'DOY', 'DATE', 'T2M', 'PRECTOTCORR']].head(10)

In [ ]:
# Check for -999 values (missing data indicator)
print("Checking for -999 values in each column:")
for col in df.columns:
    count = (df[col] == -999).sum()
    if count > 0:
        print(f"  {col}: {count} occurrences")

# Replace -999 with NaN
df = df.replace(-999, np.nan)

print("\nAfter replacing -999 with NaN:")
print(df.isnull().sum())

# Add Country column
df['Country'] = 'Ethiopia'

# Extract Month from DATE
df['Month'] = df['DATE'].dt.month

print("Data Loading Complete:")
print(f"- Dataset shape: {df.shape}")
print(f"- Date range: {df['DATE'].min()} to {df['DATE'].max()}")
print(f"- Country column added: {df['Country'].unique()}")
print(f"- Month range: {df['Month'].min()} to {df['Month'].max()}")
print("\nSample data with new columns:")
df[['YEAR', 'DOY', 'DATE', 'Country', 'Month', 'T2M', 'PRECTOTCORR']].head()

In [ ]:
# Missing Values Analysis
print("Missing Values Analysis:")
print("=" * 40)

# Count missing values after -999 replacement
missing_counts = df.isnull().sum()
missing_percentages = (missing_counts / len(df)) * 100

missing_analysis = pd.DataFrame({
    'Count': missing_counts,
    'Percentage': missing_percentages
})

print("Missing values by column:")
print(missing_analysis[missing_analysis['Count'] > 0])

# Flag columns with >5% missing values
high_missing = missing_analysis[missing_analysis['Percentage'] > 5]
print(f"\nColumns with >5% missing values: {len(high_missing)}")
if len(high_missing) > 0:
    print(high_missing)

# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates()
    print(f"Removed duplicates. New shape: {df.shape}")

print("\nDescriptive Statistics:")
print(df.describe())

In [ ]:
# Outliers Analysis using Z-scores
print("Outliers Analysis:")
print("=" * 40)

# Select key variables for outlier detection
key_vars = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'PS']

outlier_results = {}

for var in key_vars:
    if var in df.columns:
        # Calculate Z-scores
        z_scores = np.abs((df[var] - df[var].mean()) / df[var].std())
        outliers = z_scores > 3  # Standard threshold for outliers
        
        outlier_count = outliers.sum()
        outlier_percentage = (outlier_count / len(df)) * 100
        
        outlier_results[var] = {
            'count': outlier_count,
            'percentage': outlier_percentage
        }
        
        print(f"{var}: {outlier_count} outliers ({outlier_percentage:.2f}%)")

# Document outlier decisions
print("\nOutlier Treatment Decision:")
print("- Variables with >5% outliers will be carefully examined")
print("- Z-score threshold of 3 used (standard practice)")
print("- Outliers will be retained for climate analysis as they may represent extreme events")

In [ ]:
# Data Cleaning
print("Data Cleaning:")
print("=" * 40)

# Forward-fill missing values
df_clean = df.copy()
df_clean = df_clean.fillna(method='ffill')

print(f"Applied forward-fill to handle missing values")
print(f"Missing values before: {df.isnull().sum().sum()}")
print(f"Missing values after: {df_clean.isnull().sum().sum()}")

# Calculate percentage of missing values per row
missing_per_row = df.isnull().sum(axis=1) / len(df.columns)
rows_to_drop = missing_per_row > 0.3  # Rows with >30% missing values

print(f"Rows with >30% missing values: {rows_to_drop.sum()}")
df_clean = df_clean[~rows_to_drop]
print(f"Dataset shape after dropping high-missing rows: {df_clean.shape}")

# Export cleaned data
import os
if not os.path.exists('../data'):
    os.makedirs('../data')

clean_file_path = '../data/ethiopia_clean.csv'
df_clean.to_csv(clean_file_path, index=False)
print(f"Cleaned data exported to: {clean_file_path}")

# Update working dataframe
df = df_clean.copy()
print("Data cleaning completed successfully!")

In [ ]:
# Time Series Analysis
print("Time Series Analysis:")
print("=" * 40)

# Monthly average temperature analysis
monthly_temp = df.groupby('Month')['T2M'].mean()

plt.figure(figsize=(12, 6))
plt.plot(monthly_temp.index, monthly_temp.values, marker='o', linewidth=2, markersize=8)
plt.title('Ethiopia: Monthly Average Temperature (2015-2026)', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Temperature (°C)', fontsize=12)
plt.grid(True, alpha=0.3)

# Annotate warmest and coolest months
warmest_month = monthly_temp.idxmax()
coolest_month = monthly_temp.idxmin()
warmest_temp = monthly_temp.max()
coolest_temp = monthly_temp.min()

plt.annotate(f'Warmest: Month {warmest_month} ({warmest_temp:.1f}°C)', 
             xy=(warmest_month, warmest_temp), 
             xytext=(warmest_month+0.5, warmest_temp+0.5),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=10, color='red')

plt.annotate(f'Coolest: Month {coolest_month} ({coolest_temp:.1f}°C)', 
             xy=(coolest_month, coolest_temp), 
             xytext=(coolest_month-0.5, coolest_temp-0.5),
             arrowprops=dict(arrowstyle='->', color='blue'),
             fontsize=10, color='blue')

plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

print(f"Warmest month: {warmest_month} ({warmest_temp:.1f}°C)")
print(f"Coolest month: {coolest_month} ({coolest_temp:.1f}°C)")

In [ ]:
# Monthly precipitation analysis
monthly_precip = df.groupby('Month')['PRECTOTCORR'].sum()

plt.figure(figsize=(12, 6))
bars = plt.bar(monthly_precip.index, monthly_precip.values, color='skyblue', alpha=0.7)
plt.title('Ethiopia: Monthly Total Precipitation (2015-2026)', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Precipitation (mm)', fontsize=12)
plt.grid(True, alpha=0.3, axis='y')

# Annotate peak rainfall months
peak_month = monthly_precip.idxmax()
peak_precip = monthly_precip.max()

plt.annotate(f'Peak Rains: Month {peak_month} ({peak_precip:.1f}mm)', 
             xy=(peak_month, peak_precip), 
             xytext=(peak_month+0.5, peak_precip+50),
             arrowprops=dict(arrowstyle='->', color='darkblue'),
             fontsize=10, color='darkblue')

plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

print(f"Peak rainfall month: {peak_month} ({peak_precip:.1f}mm)")

# Interpretation
print("\nTime Series Interpretation:")
print(f"- Temperature shows clear seasonal patterns with {warmest_temp-coolest_temp:.1f}°C annual range")
print(f"- Precipitation peaks in month {peak_month}, indicating monsoon influence")
print("- Climate patterns typical of East African seasonal variations")

In [ ]:
# Correlation Analysis
print("Correlation Analysis:")
print("=" * 40)

# Select numerical variables for correlation
numerical_vars = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'PS']
correlation_data = df[numerical_vars].corr()

# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_data, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, fmt='.2f')
plt.title('Ethiopia: Climate Variables Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Scatter plots for key relationships
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# T2M vs RH2M
ax1.scatter(df['T2M'], df['RH2M'], alpha=0.5, s=1)
ax1.set_xlabel('Temperature (°C)')
ax1.set_ylabel('Relative Humidity (%)')
ax1.set_title('Temperature vs Relative Humidity')
ax1.grid(True, alpha=0.3)

# T2M_RANGE vs WS2M
df['T2M_RANGE'] = df['T2M_MAX'] - df['T2M_MIN']
ax2.scatter(df['T2M_RANGE'], df['WS2M'], alpha=0.5, s=1, color='orange')
ax2.set_xlabel('Temperature Range (°C)')
ax2.set_ylabel('Wind Speed (m/s)')
ax2.set_title('Temperature Range vs Wind Speed')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Top correlations
print("Top 3 correlations:")
corr_pairs = []
for i in range(len(correlation_data.columns)):
    for j in range(i+1, len(correlation_data.columns)):
        var1 = correlation_data.columns[i]
        var2 = correlation_data.columns[j]
        corr_val = correlation_data.iloc[i, j]
        if abs(corr_val) > 0.3:  # Only show meaningful correlations
            corr_pairs.append((var1, var2, corr_val))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for i, (var1, var2, corr) in enumerate(corr_pairs[:3]):
    print(f"{i+1}. {var1} vs {var2}: {corr:.3f}")

In [ ]:
# Distribution Analysis
print("Distribution Analysis:")
print("=" * 40)

# Precipitation distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
precip_data = df['PRECTOTCORR'].dropna()
if precip_data.skew() > 1:  # Use log scale if skewed
    plt.hist(precip_data, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    plt.yscale('log')
    plt.title('Precipitation Distribution (Log Scale)', fontsize=12)
    print("Precipitation distribution is highly skewed - using log scale")
else:
    plt.hist(precip_data, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    plt.title('Precipitation Distribution', fontsize=12)

plt.xlabel('Precipitation (mm)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

# Bubble chart: Temperature vs Precipitation with Wind Speed as bubble size
plt.subplot(1, 2, 2)
sample_data = df.sample(n=min(1000, len(df)))  # Sample for performance
bubble_size = sample_data['WS2M'] * 10  # Scale for visibility

plt.scatter(sample_data['T2M'], sample_data['PRECTOTCORR'], 
           s=bubble_size, alpha=0.3, c=sample_data['RH2M'], 
           cmap='viridis', edgecolors='black', linewidth=0.5)

plt.xlabel('Temperature (°C)')
plt.ylabel('Precipitation (mm)')
plt.title('Temperature vs Precipitation\n(Bubble size: Wind Speed, Color: Humidity)', fontsize=12)
plt.colorbar(label='Relative Humidity (%)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Precipitation skewness: {precip_data.skew():.2f}")
print(f"Precipitation kurtosis: {precip_data.kurtosis():.2f}")
print("\nDistribution Interpretation:")
print("- Precipitation shows right-skewed distribution typical for climate data")
print("- Bubble chart reveals complex relationships between climate variables")
print("- Higher temperatures generally associated with lower precipitation in dry seasons")